# Playground: **VUF (вектор в остатке)** vs **SAE (латентный bump)**

На одних промптах сравниваем три режима: **baseline** (без хуков), **сырой VUF** как в Meta (`h ← h + α·r̂` из `Hs_hedge_universal.pt`), **SAE-интервенцию** (`encode → f+αδ → decode + error` из `intervention.pt`).

Нужны GPU и **Hugging Face** (Mistral — gated). Файл **`Hs_hedge_universal.pt`** кладите в **`vuf_vectors/`** внутри клона (каталог создаётся в §2); если файла там нет, в §4 проверяется запасной путь `calibration/outputs/merged/...` после копирования чекпоинта. Для SAE — **`mistral_intervention.pt`** из `build_intervention_config` или оставьте `None` (случайный `delta` только для проверки хуков).

**Код `sae_muc/`** клонируется в §1. URL: `SAE_MUC_GIT_URL`.

**NumPy / transformers:** при `dtype size changed` — *Restart runtime*, затем только §1, потом остальное.

**HF:** §3 — `HF_TOKEN` в Secrets или `login()`.

## 1. Клон репозитория с GitHub + установка зависимостей

По времени: клон — секунды; **pip (особенно force-reinstall transformers)** — часто **3–10 минут**, зависит от Colab и сети. В ячейке **полный лог pip** (без `-q`), чтобы было видно, что не зависло.

In [ ]:
import os, sys, subprocess

# Репозиторий с корнем вида: .../sae_muc/__init__.py (например github.com/SadreevAmir/sae-muc)
GIT_URL = os.environ.get("SAE_MUC_GIT_URL", "https://github.com/SadreevAmir/sae-muc.git")
GIT_BRANCH = os.environ.get("SAE_MUC_BRANCH", "main")
REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    parent = os.path.dirname(REPO_DIR.rstrip("/")) or "/content"
    os.makedirs(parent, exist_ok=True)
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

assert os.path.isdir(sae_pkg), f"После клона ожидается {sae_pkg}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# PyTorch не переустанавливаем. Сброс NumPy + force-reinstall transformers (dtype size changed в Colab).
print("→ pip uninstall numpy …")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
print("→ pip install numpy (force-reinstall, может 1–2 мин) …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", "--force-reinstall",
    "numpy>=2.0.0,<2.1",
])
print("→ pip install transformers + accelerate (force-reinstall, часто долго) …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "--force-reinstall",
    "transformers>=4.40", "accelerate",
])
print("→ pip install sae-lens и остальное …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U",
    "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub",
])
subprocess.check_call([
    sys.executable, "-c",
    "import numpy, numpy.random; print('numpy OK', numpy.__version__)",
])

import torch
assert torch.cuda.is_available(), "Нужен GPU runtime"

_vuf_hooks_py = os.path.join(sae_pkg, "vuf_hooks.py")
assert os.path.isfile(_vuf_hooks_py), (
    f"Нет {_vuf_hooks_py}. Выполните git pull в {REPO_DIR} или переклонируйте репозиторий "
    "(в main должен быть sae_muc/vuf_hooks.py)."
)

print("REPO_DIR:", REPO_DIR)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Конфиг

**Mistral** + `mistral-7b-res-wg` по умолчанию. Для **Llama Scope** — раскомментируй блок Llama в коде.

- **`VUF_DIR`** — папка под VUF (`…/sae-muc/vuf_vectors`); создаётся автоматически. Сюда положите **`Hs_hedge_universal.pt`** (или задайте `VUF_HEDGE_PATH` вручную).
- **`VUF_HEDGE_FALLBACK`** — если в `VUF_DIR` файла нет, в §4 пробуется путь из дерева `calibration/…` (как после полного чекпоинта).
- **`INTERVENTION_PT`** — выход `python -m sae_muc.build_intervention_config ...`; `None` — случайный SAE-`delta` (только smoke-test).
- **`ALPHA_VUF` / `ALPHA_SAE`** — силы для двух методов (можно разные).

In [ ]:
import os

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

# Папка под VUF-вектор: положите сюда Hs_hedge_universal.pt (удобно залить с Drive / скопировать один файл)
VUF_DIR = os.path.join(REPO_DIR, "vuf_vectors")
os.makedirs(VUF_DIR, exist_ok=True)

# --- Дефолт: Mistral 7B Instruct + mistral-7b-res-wg ---
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
SAE_RELEASE = "mistral-7b-res-wg"
PROCESS_LAYERS = [15, 23]

# --- Опционально: Llama 3.1 + Llama Scope ---
# MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# SAE_RELEASE = "llama_scope_lxr_32x"
# PROCESS_LAYERS = [15, 23]

VUF_HEDGE_PATH = os.path.join(VUF_DIR, "Hs_hedge_universal.pt")
# Если держите полное дерево calibration после пайплайна — §4 подхватит этот путь, если в vuf_vectors файла нет:
VUF_HEDGE_FALLBACK = os.path.join(
    REPO_DIR,
    "calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt",
)
# VUF_HEDGE_PATH = None  # только SAE + baseline (и отключит и fallback, если тоже обнулить ниже)
# VUF_HEDGE_FALLBACK = None

# SAE: артефакт build_intervention_config; None = случайный delta (не для сравнения с продом)
INTERVENTION_PT = None  # os.path.join(REPO_DIR, "sae_muc/artifacts/mistral_intervention.pt")

ALPHA_VUF = 1.0   # как типичный max_alpha в MUC (подберите 0.3–2)
ALPHA_SAE = 2.0   # сила в SAE-пространстве (часто другой масштаб, чем VUF)

SAE_DTYPE = "float32"

print("VUF_DIR (положите сюда Hs_hedge_universal.pt):", VUF_DIR)

## 3. Hugging Face (для gated моделей)

Токен с [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (read). В Colab: **Secrets** → `HF_TOKEN`, либо вставка через `login()`.

In [ ]:
import os
from huggingface_hub import login

tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if tok:
    login(token=tok, add_to_git_credential=False)
else:
    login()  # вставь токен вручную

## 4. Модель, `Hs_hedge` (VUF) и SAE

In [ ]:
import os, sys
_REPO = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)
if not os.path.isdir(os.path.join(_REPO, "sae_muc")):
    raise RuntimeError(
        f"Нет каталога sae_muc в {_REPO}. Выполни ячейку §1 (clone + pip), затем эту снова."
    )

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

from sae_muc.hooks import clear_sae_latent_hooks, register_sae_latent_hooks
from sae_muc.vuf_hooks import clear_vuf_residual_hooks, register_vuf_residual_hooks

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id


def load_intervention_or_random(intervention_path, release, process_layers, sae_dtype):
    layer_to_sae = {}
    layer_to_delta = {}
    if intervention_path and os.path.isfile(intervention_path):
        blob = torch.load(intervention_path, map_location="cpu")
        release = blob["release"]
        for k, meta in blob["layers"].items():
            hf_layer = int(k)
            if hf_layer not in process_layers:
                continue
            sid = meta["sae_id"]
            print(f"SAE layer {hf_layer} <- {sid}")
            layer_to_sae[hf_layer] = SAE.from_pretrained(
                release, sid, device="cpu", dtype=sae_dtype
            )
            layer_to_delta[hf_layer] = meta["delta"]
    else:
        print("Нет INTERVENTION_PT — случайный delta (только проверка SAE-хуков)")
        from sae_muc.layer_map import hf_layers_for_release

        mapping = {hf: sid for hf, sid in hf_layers_for_release(release)}
        for L in process_layers:
            if L not in mapping:
                continue
            sid = mapping[L]
            sae = SAE.from_pretrained(release, sid, device="cpu", dtype=sae_dtype)
            g = torch.randn(sae.cfg.d_sae, dtype=torch.float32)
            g = g / (g.norm() + 1e-8)
            layer_to_sae[L] = sae
            layer_to_delta[L] = g
    return layer_to_sae, layer_to_delta


hedge_2d = None
_vuf_fallback = globals().get("VUF_HEDGE_FALLBACK")
_vuf_paths = [p for p in (VUF_HEDGE_PATH, _vuf_fallback) if p]
_loaded_from = None
for p in _vuf_paths:
    if os.path.isfile(p):
        hedge_2d = torch.load(p, map_location="cpu")
        _loaded_from = p
        break
if hedge_2d is not None:
    if hedge_2d.ndim != 2:
        raise ValueError(f"Hs_hedge ожидается [n_layers, d_model], got {tuple(hedge_2d.shape)}")
    print("VUF Hs_hedge:", tuple(hedge_2d.shape), "←", _loaded_from)
elif _vuf_paths:
    print("VUF: файла нет. Положите Hs_hedge_universal.pt в VUF_DIR или calibration/… Проверялись:", _vuf_paths)
else:
    print("VUF: пути отключены — колонка «VUF» в §5 будет пропущена.")

layer_to_sae, layer_to_delta = load_intervention_or_random(
    INTERVENTION_PT, SAE_RELEASE, PROCESS_LAYERS, SAE_DTYPE
)
print("Загружено слоёв с SAE:", list(layer_to_sae.keys()))

## 4b. Какие латенты трогаем (Neuronpedia)

Модуль `sae_muc.inspect_delta` печатает **индексы ненулевых** координат в `delta` — это номера признаков SAE.

**Размеченные** страницы (autointerp, топ-примеры) — на [Neuronpedia](https://www.neuronpedia.org). Нужны **`MODEL_ID`** и **`SAE_ID` из URL** сайта (они **не** совпадают с id SAELens вроде `blocks.16.hook_resid_pre`).

Автоссылки Neuronpedia есть для **`llama_scope_lxr_32x`**. Для **Mistral** (`mistral-7b-res-wg`) задай `NP_SAE_BY_LAYER` вручную по URL на сайте или смотри только таблицу индексов.

In [ ]:
import os, sys
_REPO = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)

from sae_muc.inspect_delta import (
    neuronpedia_feature_urls,
    print_sparse_delta_report,
    print_top_explanations,
)
from sae_muc.layer_map import neuronpedia_residual_slug

# Переопределение вручную (из URL Neuronpedia), если сменил SAE_RELEASE:
NP_SAE_BY_LAYER = {}  # напр. {15: "15-llamascope-res-131k"}

for L, d in layer_to_delta.items():
    print_sparse_delta_report(L, d, limit=35)
    auto = neuronpedia_residual_slug(SAE_RELEASE, L)
    np_model = os.environ.get("NEURONPEDIA_MODEL", "") or (auto[0] if auto else "")
    slug = (
        NP_SAE_BY_LAYER.get(L)
        or os.environ.get(f"NEURONPEDIA_SAE_LAYER_{L}", "")
        or (auto[1] if auto else "")
    )
    if np_model and slug:
        print("Ссылки Neuronpedia:")
        for u in neuronpedia_feature_urls(np_model, slug, d, limit=12):
            print(" ", u)
        print("Explanations (API, если доступно):")
        print_top_explanations(np_model, slug, d, limit=5)
    else:
        print("Нет автосопоставления Neuronpedia для этого SAE_RELEASE — задай NP_SAE_BY_LAYER / env.")
    print()

## 5. Генерация: baseline → VUF (остаток) → SAE

Перед каждым прогоном снимаются **оба** типа хуков. Если `hedge_2d` нет — строка VUF пропускается.

In [ ]:
def build_inputs(user_text: str, system: str | None = None):
    if system:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ]
    else:
        messages = [{"role": "user", "content": user_text}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)


@torch.inference_mode()
def generate_one(user_text: str, max_new_tokens: int = 128, temperature: float = 0.7, do_sample: bool = True):
    enc = build_inputs(user_text)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[0, enc["input_ids"].shape[1] :]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


def compare_three(
    user_text: str,
    alpha_vuf: float | None = None,
    alpha_sae: float | None = None,
    **gen_kw,
):
    av = ALPHA_VUF if alpha_vuf is None else alpha_vuf
    as_ = ALPHA_SAE if alpha_sae is None else alpha_sae
    clear_sae_latent_hooks(model)
    clear_vuf_residual_hooks(model)
    base = generate_one(user_text, **gen_kw)
    vuf_out = None
    if hedge_2d is not None:
        clear_sae_latent_hooks(model)
        clear_vuf_residual_hooks(model)
        register_vuf_residual_hooks(model, hedge_2d, PROCESS_LAYERS, float(av))
        vuf_out = generate_one(user_text, **gen_kw)
        clear_vuf_residual_hooks(model)
    clear_sae_latent_hooks(model)
    clear_vuf_residual_hooks(model)
    register_sae_latent_hooks(model, layer_to_sae, layer_to_delta, PROCESS_LAYERS, float(as_))
    sae_out = generate_one(user_text, **gen_kw)
    clear_sae_latent_hooks(model)
    return base, vuf_out, sae_out


def show_triple(title: str, user_text: str, alpha_vuf: float | None = None, alpha_sae: float | None = None, **gen_kw):
    b, v, s = compare_three(user_text, alpha_vuf=alpha_vuf, alpha_sae=alpha_sae, **gen_kw)
    av = ALPHA_VUF if alpha_vuf is None else alpha_vuf
    as_ = ALPHA_SAE if alpha_sae is None else alpha_sae
    print("=" * 60)
    print(title)
    print("Q:", user_text[:200], "..." if len(user_text) > 200 else "")
    print("-" * 60)
    print("[baseline]\n", b)
    print("-" * 60)
    if v is None:
        print("[VUF] (нет Hs_hedge)")
    else:
        print(f"[VUF: α={av}, слои {PROCESS_LAYERS}]\n", v)
    print("-" * 60)
    print(f"[SAE: α={as_}, слои {PROCESS_LAYERS}]\n", s)
    print()

## 6. Примеры (замените на свои вопросы)

In [ ]:
show_triple(
    "Факт",
    "What is the 29th largest city in England? Answer in one short phrase.",
    do_sample=False,
    temperature=0.1,
    max_new_tokens=80,
)

show_triple(
    "Осторожная формулировка",
    "Are you completely sure about factual claims? Reply briefly.",
    max_new_tokens=100,
)

## 7. Подбор α (VUF и SAE отдельно)

Ниже: фиксированные `ALPHA_VUF` / `ALPHA_SAE` из §2. Чтобы сканировать сетку — временно меняйте их в §2 или оберните цикл.

In [ ]:
PROMPT = "Name one thing you are uncertain about regarding the year 2150. One sentence."
b, v, s = compare_three(PROMPT, max_new_tokens=80, do_sample=False)
print("baseline:", b[:200], "…" if len(b) > 200 else "")
if v is not None:
    print(f"VUF (α={ALPHA_VUF}):", v[:200], "…" if len(v) > 200 else "")
print(f"SAE (α={ALPHA_SAE}):", s[:200], "…" if len(s) > 200 else "")

print("\n--- Скан SAE-α (VUF и baseline из §2) ---")
for a_sae in [0.5, 1.0, 2.0, 4.0]:
    clear_sae_latent_hooks(model)
    clear_vuf_residual_hooks(model)
    _, _, t = compare_three(PROMPT, alpha_sae=a_sae, max_new_tokens=80, do_sample=False)
    print(f"SAE α={a_sae}: {t!r}")